In [39]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVC, SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import warnings
import logging

from pcntoolkit import (
    BLR,
    BsplineBasisFunction,
    LinearBasisFunction,
    NormativeModel,
    NormData,
    load_fcon1000,
    plot_centiles_advanced,
    plot_qq,
    plot_ridge,
)

import pcntoolkit.util.output
import seaborn as sns

sns.set_style("darkgrid")

# Suppress some annoying warnings and logs
pymc_logger = logging.getLogger("pymc")

pymc_logger.setLevel(logging.WARNING)
pymc_logger.propagate = False

warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None  # default='warn'
pcntoolkit.util.output.Output.set_show_messages(True)

In [2]:
path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/BLR/Final_BLRs/Final_BLR_corrected_results"
model = NormativeModel.load(path)

In [40]:
#full_data = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/final_data/Full_Dataset_UKB_ADNI_OASIS3.csv")
full_data = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Data/unseen_data_ukb_adni_o3.csv")
full_data = full_data[full_data["Diagnosis"] != 0]

data_df = full_data.iloc[:,1:]
demographic_data = full_data.iloc[:,:5]

# data_df

# data_df = data_df[~data_df["Site"].isin([9.0, 10.0, 14.0, 16.0, 19.0, 20.0, 24.0, 27.0, 35.0, 51.0, 57.0, 68.0, 62.0, 67.0, 70.0, 114.0, 130.0, 133.0, 135.0, 136.0, 141.0, 153.0])]
# data_df = data_df[data_df["Site"].isin([128, 129, 2, 3, 6, 137, 11, 12, 13, 401, 11025, 11026, 11027, 403, 402, 405, 404, 11028, 23, 18, 22, 32, 33, 36, 37, 41, 941, 72, 73, 82, 99, 100, 109, 116, 123])]
# #removing these ages because they had unique site & age combinations which could not be split 
# data_df = data_df[data_df["Age"] != 96.0]
# data_df = data_df[data_df["Age"] != 91.0]
# data_df = data_df[data_df["Age"] != 95.0]
# data_df = data_df[data_df["Age"] != 97.0]

# data = data_df
# covariates = ["Age"]
# batch_effects = ["Sex", "Site"]

# data_df = data_df.drop(columns=['Age','Sex','Site','Dataset', "Diagnosis"])
# columns = list(data_df.columns)

# response_vars = columns

# # create a NormData object
# norm_data = NormData.from_dataframe(
#     name="Final_BLR_corrected", dataframe=data, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars
# )
# norm_data.coords

# predictions = model.predict(norm_data)


In [41]:
norm_data_copy = norm_data
norm_data.data_vars


Data variables:
    subject_ids    (observations) int64 6kB 0 1 2 3 4 5 ... 760 761 762 763 764
    Y              (observations, response_vars) float64 630kB 2.477 ... 2.00...
    X              (observations, covariates) float64 6kB 65.0 71.0 ... 71.0
    batch_effects  (observations, batch_effect_dims) <U32 196kB '0.0' ... '40...
    Z              (observations, response_vars) float64 630kB -0.3298 ... -0...
    centiles       (centile, observations, response_vars) float64 3MB 1.905 ....
    logp           (observations, response_vars) float64 630kB -1.341 ... -0....
    Yhat           (observations, response_vars) float64 630kB 2.509 ... 4.21...
    statistics     (response_vars, statistic) float64 9kB 0.02783 ... 0.9477
    Y_harmonized   (observations, response_vars) float64 630kB 2.744 ... 2.39...

In [42]:
predictions_copy = predictions
predictions.data_vars


Data variables:
    subject_ids    (observations) int64 6kB 0 1 2 3 4 5 ... 760 761 762 763 764
    Y              (observations, response_vars) float64 630kB 2.477 ... 2.00...
    X              (observations, covariates) float64 6kB 65.0 71.0 ... 71.0
    batch_effects  (observations, batch_effect_dims) <U32 196kB '0.0' ... '40...
    Z              (observations, response_vars) float64 630kB -0.3298 ... -0...
    centiles       (centile, observations, response_vars) float64 3MB 1.905 ....
    logp           (observations, response_vars) float64 630kB -1.341 ... -0....
    Yhat           (observations, response_vars) float64 630kB 2.509 ... 4.21...
    statistics     (response_vars, statistic) float64 9kB 0.02783 ... 0.9477
    Y_harmonized   (observations, response_vars) float64 630kB 2.744 ... 2.39...

In [43]:
z_scores = predictions["Z"]
z_scores


<xarray.DataArray 'Z' (observations: 765, response_vars: 103)> Size: 630kB
array([[-3.29794837e-01, -5.49093829e-01, -2.11055961e-01, ...,
         2.61529540e-01, -6.10009633e-01,  1.09796770e+00],
       [ 1.85626442e-01, -2.21268856e-01,  2.31759408e-01, ...,
         1.86144206e+00,  9.59222831e-01,  5.12751298e-01],
       [ 3.05913276e-01, -5.34513519e-01, -3.25007260e-01, ...,
         3.82702471e-01,  3.36945319e+00,  3.09253237e+00],
       ...,
       [ 6.70080149e-01, -6.10099249e-02, -5.68412106e-03, ...,
         6.38897045e-01,  3.49996813e-01,  7.76976009e-01],
       [-1.40412713e-01, -7.13679969e-01, -2.17385118e-01, ...,
        -3.13824368e-01,  7.15926942e-01,  4.03521302e-02],
       [-5.44132847e-01, -2.56373208e-01,  1.37154614e-01, ...,
         2.37144610e-03,  3.75815102e-01, -5.78328718e-01]],
      shape=(765, 103))
Coordinates:
  * observations   (observations) int64 6kB 0 1 2 3 4 5 ... 760 761 762 763 764
  * response_vars  (response_vars) <U34 14kB 'lh_G_front_inf-Triangul_thickne...

In [44]:
z_scores_df = z_scores.to_pandas()
z_scores_df

response_vars,lh_G_front_inf-Triangul_thickness,lh_G_front_middle_thickness,lh_G_front_sup_thickness,lh_G_oc-temp_lat-fusifor_thickness,lh_G_oc-temp_med-Lingual_thickness,lh_G_oc-temp_med-Parahip_thickness,lh_G_occipital_middle_thickness,lh_G_orbital_thickness,lh_G_pariet_inf-Angular_thickness,lh_G_pariet_inf-Supramar_thickness,...,Right-Cerebellum-White-Matter,Right-Hippocampus,Right-Inf-Lat-Vent,Right-Lateral-Ventricle,Right-Pallidum,Right-Putamen,Right-VentralDC,Right-choroid-plexus,Right-vessel,WM-hypointensities
observations,,,,,,,,,,,,,,,,,,,,,
0,-0.329795,-0.549094,-0.211056,0.406967,1.009857,-1.478036,-1.045472,-0.039979,-1.024370,-0.885151,...,-1.108435,-1.397909,1.039118,-0.413638,-0.693457,-0.204875,-1.551464,0.261530,-0.610010,1.097968
1,0.185626,-0.221269,0.231759,-0.056082,0.407224,-0.114062,-0.277771,0.040912,0.838708,-0.339521,...,0.801529,-0.988845,0.664508,-0.914622,0.681383,0.190109,-0.538984,1.861442,0.959223,0.512751
2,0.305913,-0.534514,-0.325007,-0.573426,0.121846,-0.664188,-0.877435,-0.106262,0.099461,-0.586271,...,-0.275950,-1.278759,-0.344274,-1.018247,-0.297784,-1.121141,-1.377735,0.382702,3.369453,3.092532
3,-0.561590,-0.639036,-0.060851,0.511599,0.398564,-0.352792,-0.635084,0.410430,-0.614148,-0.582446,...,1.964870,-0.220213,-0.201895,-0.313894,-0.229633,-0.486843,-0.150476,-0.230506,-0.362630,0.088609
4,-0.668011,-0.763320,-0.612880,-1.237602,-0.191958,0.268895,-0.499685,-0.220294,-1.005737,-0.909795,...,1.377315,1.883172,0.731602,0.491081,0.768602,0.350655,1.212735,0.271038,-0.787955,-0.085306
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,0.724821,0.086462,-0.167231,-0.720738,-0.472790,-1.102290,0.119921,-0.555356,0.772815,-0.501094,...,-0.672189,-0.424295,3.854293,2.970787,0.029159,0.981549,0.926234,2.113449,0.746364,0.993805
761,0.521586,-0.256928,-0.087539,-0.956958,-0.862477,0.063466,-1.395441,-0.046261,-0.253079,-0.308713,...,-1.089389,-1.995476,1.292303,2.332390,-0.057991,-1.081574,-0.954406,0.138043,-0.847034,0.105780
762,0.670080,-0.061010,-0.005684,0.536079,1.869902,-0.272032,0.152777,-0.211363,0.363847,0.167016,...,0.132157,-1.163179,1.213236,0.352739,1.871425,0.356163,0.737623,0.638897,0.349997,0.776976


In [46]:
demographic_data = demographic_data.iloc[:,1:]
z_scores_df = z_scores_df.iloc[:,1:]
demographic_data

,Age,Sex,Site,Diagnosis
3533,65.0,0.0,2.0,1.0
3534,71.0,1.0,2.0,2.0
3535,69.0,0.0,2.0,1.0
3536,71.0,0.0,2.0,2.0
3537,74.0,1.0,2.0,1.0
...,...,...,...,...
5406,69.0,0.0,402.0,1.0
5407,68.0,0.0,403.0,1.0
5408,71.0,1.0,403.0,1.0
5409,73.0,1.0,401.0,1.0


In [50]:
z_scored_data = pd.concat([demographic_data, z_scores_df], axis=0)

z_scores_df

response_vars,lh_G_front_sup_thickness,lh_G_oc-temp_lat-fusifor_thickness,lh_G_oc-temp_med-Lingual_thickness,lh_G_oc-temp_med-Parahip_thickness,lh_G_occipital_middle_thickness,lh_G_orbital_thickness,lh_G_pariet_inf-Angular_thickness,lh_G_pariet_inf-Supramar_thickness,lh_G_parietal_sup_thickness,lh_G_postcentral_thickness,...,Right-Cerebellum-White-Matter,Right-Hippocampus,Right-Inf-Lat-Vent,Right-Lateral-Ventricle,Right-Pallidum,Right-Putamen,Right-VentralDC,Right-choroid-plexus,Right-vessel,WM-hypointensities
observations,,,,,,,,,,,,,,,,,,,,,
0,-0.211056,0.406967,1.009857,-1.478036,-1.045472,-0.039979,-1.024370,-0.885151,-0.572125,-0.588868,...,-1.108435,-1.397909,1.039118,-0.413638,-0.693457,-0.204875,-1.551464,0.261530,-0.610010,1.097968
1,0.231759,-0.056082,0.407224,-0.114062,-0.277771,0.040912,0.838708,-0.339521,2.312108,0.416639,...,0.801529,-0.988845,0.664508,-0.914622,0.681383,0.190109,-0.538984,1.861442,0.959223,0.512751
2,-0.325007,-0.573426,0.121846,-0.664188,-0.877435,-0.106262,0.099461,-0.586271,-0.428195,-0.862259,...,-0.275950,-1.278759,-0.344274,-1.018247,-0.297784,-1.121141,-1.377735,0.382702,3.369453,3.092532
3,-0.060851,0.511599,0.398564,-0.352792,-0.635084,0.410430,-0.614148,-0.582446,-0.490714,-0.347661,...,1.964870,-0.220213,-0.201895,-0.313894,-0.229633,-0.486843,-0.150476,-0.230506,-0.362630,0.088609
4,-0.612880,-1.237602,-0.191958,0.268895,-0.499685,-0.220294,-1.005737,-0.909795,-1.005660,-0.777586,...,1.377315,1.883172,0.731602,0.491081,0.768602,0.350655,1.212735,0.271038,-0.787955,-0.085306
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,-0.167231,-0.720738,-0.472790,-1.102290,0.119921,-0.555356,0.772815,-0.501094,0.820011,2.538154,...,-0.672189,-0.424295,3.854293,2.970787,0.029159,0.981549,0.926234,2.113449,0.746364,0.993805
761,-0.087539,-0.956958,-0.862477,0.063466,-1.395441,-0.046261,-0.253079,-0.308713,-0.728123,-0.643278,...,-1.089389,-1.995476,1.292303,2.332390,-0.057991,-1.081574,-0.954406,0.138043,-0.847034,0.105780
762,-0.005684,0.536079,1.869902,-0.272032,0.152777,-0.211363,0.363847,0.167016,0.165232,0.066181,...,0.132157,-1.163179,1.213236,0.352739,1.871425,0.356163,0.737623,0.638897,0.349997,0.776976


In [11]:
# demographic_data = demographic_data[~demographic_data["Site"].isin([9.0, 10.0, 14.0, 16.0, 19.0, 20.0, 24.0, 27.0, 35.0, 51.0, 57.0, 68.0, 62.0, 67.0, 70.0, 114.0, 130.0, 133.0, 135.0, 136.0, 141.0, 153.0])]
# demographic_data = demographic_data[demographic_data["Site"].isin([128, 129, 2, 3, 6, 137, 11, 12, 13, 401, 11025, 11026, 11027, 403, 402, 405, 404, 11028, 23, 18, 22, 32, 33, 36, 37, 41, 941, 72, 73, 82, 99, 100, 109, 116, 123])]
# #removing these ages because they had unique site & age combinations which could not be split 
# demographic_data = demographic_data[demographic_data["Age"] != 96.0]
# demographic_data = demographic_data[demographic_data["Age"] != 91.0]
# demographic_data = demographic_data[demographic_data["Age"] != 95.0]
# demographic_data = demographic_data[demographic_data["Age"] != 97.0]

# demographic_data.shape #(36593, 5)
# #z_scores_df.shape # (36179, 103)
# z_scored_data = pd.concat([demographic_data, z_scores_df], axis=1)
# z_scored_data

z_scores = predictions["Z"]
age = predictions["X"]
batch_effects = predictions["batch_effects"]

z_scores_df = z_scores.to_pandas()
age_df = age.to_pandas()
batch_effects_df = batch_effects.to_pandas()

z_scored_data = pd.merge([age_df, batch_effects_df, z_scores_df], on="observations", how="left")
z_scored_data

TypeError: merge() missing 1 required positional argument: 'right'

In [12]:
df = predictions.to_dataframe()
df

X             Y                                                  \
      Age 3rd-Ventricle 4th-Ventricle Brain-Stem CC_Anterior CC_Central   
0    65.0        1092.2        1894.9    17589.6       777.2      402.8   
1    71.0        1667.0        1662.2    21435.9       800.9      470.7   
2    69.0         836.5        2438.9    17935.8       793.1      380.7   
3    71.0         952.2        2237.3    23312.4       796.2      368.7   
4    74.0        2207.1        2526.6    23231.0       937.4      426.6   
..    ...           ...           ...        ...         ...        ...   
760  69.0        2901.0        3086.5    19062.8       788.2      338.0   
761  68.0        2248.0        2044.0    16662.4       595.2      245.6   
762  71.0        3379.3        1482.7    21591.6       774.0      313.9   
763  73.0        1246.6        2147.6    18254.5       815.8      374.1   
764  71.0        1245.6        2124.2    18019.3       707.0      350.8   

                                                           ...  \
    CC_Mid_Anterior CC_Mid_Posterior CC_Posterior     CSF  ...   
0             492.4            501.0        983.5  1042.3  ...   
1             444.9            413.2        899.4  1501.5  ...   
2             385.4            363.9        926.2   911.8  ...   
3             351.5            404.0        950.7   727.0  ...   
4             421.1            475.5       1065.3  1234.8  ...   
..              ...              ...          ...     ...  ...   
760           307.9            223.5        817.8  1278.1  ...   
761           240.1            201.9        655.9  2264.1  ...   
762           329.3            294.8        705.3  1370.1  ...   
763           394.4            339.3        766.3  1108.1  ...   
764           348.2            293.1        904.0  1158.1  ...   

                                 centiles                            \
    (Right-Cerebellum-White-Matter, 0.95) (Right-Hippocampus, 0.95)   
0                            16118.303985               4497.942198   
1                            16884.545959               4599.514531   
2                            15904.002517               4379.973349   
3                            15815.808793               4322.416960   
4                            16728.223188               4502.071813   
..                                    ...                       ...   
760                          16581.608182               4454.478400   
761                          17065.572073               4471.913441   
762                          18298.361995               4673.788520   
763                          17946.479650               4596.656004   
764                          16732.204404               4376.080718   

                                                                \
    (Right-Inf-Lat-Vent, 0.95) (Right-Lateral-Ventricle, 0.95)   
0                   850.000730                    20639.927886   
1                  1483.544323                    34634.164279   
2                  1042.615962                    24175.552084   
3                  1169.591766                    26390.972658   
4                  1808.421595                    40744.428530   
..                         ...                             ...   
760                 838.627764                    20801.892561   
761                 835.247955                    20863.333347   
762                1232.725327                    30519.167926   
763                1363.322207                    33290.577497   
764                 959.911552                    23290.198762   

                                                                          \
    (Right-Pallidum, 0.95) (Right-Putamen, 0.95) (Right-VentralDC, 0.95)   
0              2181.420731           5358.605540             4134.685839   
1              2381.955563           5834.075663             4505.076558   
2              2172.705695           5278.119139             4081.507525   
3         

In [13]:
df_age = df["X"]
df_age

df_z = df["Z"]
df_z

df_batch = df["batch_effects"]
df_batch

df_data = pd.concat([df_age, df_batch, df_z],axis=1)
df_data.shape
df_data

,Age,Sex,Site,3rd-Ventricle,4th-Ventricle,Brain-Stem,CC_Anterior,CC_Central,CC_Mid_Anterior,CC_Mid_Posterior,...,rh_S_front_sup_thickness,rh_S_interm_prim-Jensen_thickness,rh_S_oc-temp_lat_thickness,rh_S_occipital_ant_thickness,rh_S_parieto_occipital_thickness,rh_S_pericallosal_thickness,rh_S_postcentral_thickness,rh_S_precentral-inf-part_thickness,rh_S_precentral-sup-part_thickness,rh_S_temporal_sup_thickness
0,65.0,0.0,2.0,-0.188814,0.386875,-0.922409,-0.833817,-0.771080,0.199928,-0.247645,...,0.102813,0.234268,-0.533951,-0.698476,-0.526538,1.768101,-0.512109,-0.233694,-0.310310,-0.547945
1,71.0,1.0,2.0,-0.034683,-0.689506,-0.294852,-0.833265,0.112480,-0.228847,-1.014119,...,-0.379661,0.153262,-0.633351,-0.210950,-0.388447,0.504387,0.480799,-0.077158,0.338037,-0.179143
2,69.0,0.0,2.0,-0.993462,1.311986,-0.714176,-0.632579,-0.854503,-0.697929,-1.335541,...,-0.778378,-0.131248,-0.630640,-0.109359,-0.974146,1.178048,-0.553657,-1.309408,-0.154326,-0.325221
3,71.0,0.0,2.0,-0.862121,0.964566,1.619521,-0.552468,-0.910804,-0.921034,-0.975334,...,-0.387532,0.321393,0.479319,-0.325803,-0.185142,0.459679,-0.295419,0.041264,-0.182481,-0.268305
4,74.0,1.0,2.0,0.607778,0.811242,0.613447,0.247986,-0.313326,-0.396674,-0.338758,...,-0.595086,-0.557081,-0.529221,-0.546449,-0.422984,-0.402075,-0.735155,-0.044867,-0.406847,-0.506943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
760,69.0,0.0,402.0,2.726357,2.229415,0.033126,0.099835,-0.392672,-0.504101,-0.795217,...,-0.038506,-0.658395,-0.499578,-0.363305,0.176552,-0.347946,0.813132,-0.297906,0.708942,-0.176879
761,68.0,0.0,403.0,2.129675,0.752902,-0.959432,-0.949545,-1.089027,-0.968966,-0.935712,...,-0.322103,-1.401442,-1.374746,-0.938304,-0.750469,-0.028721,-0.680373,-0.031459,-0.805253,-0.544280
762,71.0,1.0,403.0,1.992183,-1.031909,-0.027629,-0.278011,-0.597256,-0.464361,-0.477883,...,0.175084,0.132413,0.067906,0.543355,0.404912,-0.141960,0.604250,0.176402,0.603165,-0.260271
763,73.0,1.0,401.0,-0.996167,0.264438,-1.474771,0.047846,-0.044953,0.094292,-0.154623,...,-0.249686,-0.517016,-1.832093,-1.833396,-1.042145,-0.853699,-0.894481,-0.316510,-0.835310,-1.116426


In [17]:
# check_df = pd.concat([df_age, df_batch, demographic_data],axis=1)
# check_df
full_data = pd.read_csv("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Data/unseen_data_ukb_adni_o3.csv")
data_df = full_data.iloc[:,1:]
demographic_data = full_data.iloc[:,:5]
demographic_data = demographic_data[~demographic_data["Site"].isin([9.0, 10.0, 14.0, 16.0, 19.0, 20.0, 24.0, 27.0, 35.0, 51.0, 57.0, 68.0, 62.0, 67.0, 70.0, 114.0, 130.0, 133.0, 135.0, 136.0, 141.0, 153.0])]
demographic_data = demographic_data[demographic_data["Site"].isin([128, 129, 2, 3, 6, 137, 11, 12, 13, 401, 11025, 11026, 11027, 403, 402, 405, 404, 11028, 23, 18, 22, 32, 33, 36, 37, 41, 941, 72, 73, 82, 99, 100, 109, 116, 123])]
#removing these ages because they had unique site & age combinations which could not be split 
demographic_data = demographic_data[demographic_data["Age"] != 96.0]
demographic_data = demographic_data[demographic_data["Age"] != 91.0]
demographic_data = demographic_data[demographic_data["Age"] != 95.0]
demographic_data = demographic_data[demographic_data["Age"] != 97.0]

demographic_data = demographic_data.iloc[:,1:]
demographic_data

,Age,Sex,Site,Diagnosis
0,58.0,0.0,11025.0,0.0
1,68.0,1.0,11025.0,0.0
2,63.0,1.0,11025.0,0.0
3,52.0,0.0,11025.0,0.0
4,56.0,0.0,11025.0,0.0
...,...,...,...,...
5406,69.0,0.0,402.0,1.0
5407,68.0,0.0,403.0,1.0
5408,71.0,1.0,403.0,1.0
5409,73.0,1.0,401.0,1.0


In [51]:
df_age = df["X"].reset_index(drop=True)
df_z = df["Z"].reset_index(drop=True)
df_batch = df["batch_effects"].reset_index(drop=True)
df_data = pd.concat([df_age, df_batch], axis=1)
df_diag = demographic_data["Diagnosis"].reset_index(drop=True)
df_diag_data = pd.concat([df_data, df_diag, df_z], axis=1)



In [55]:
df_diag_data = df_diag_data.dropna()
df_diag_data.to_csv("test_zscores_brainplot.csv")